Localizamos el Repo y las rutas

In [10]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)

ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated


Compruebo spark y que existe open jdk

In [11]:
import pyspark, shutil, subprocess

print("pyspark", pyspark.__version__)
print(subprocess.check_output(["java", "-version"], text=True, stderr=subprocess.STDOUT).splitlines()[0])
print("java:", shutil.which("java"))

pyspark 3.5.5
openjdk version "17.0.20.1" 2026-08-18
java: /usr/bin/java


Creamos una sesión local de spark

In [12]:
spark = get_spark("novashop-m01")
spark

Vamos a crear un Dataframe

In [13]:
from pyspark.sql import Row

pedidos = [
    Row(order_id="O90001", customer_id="C0001", status="paid", amount=49.90),
    Row(order_id="O90002", customer_id="C0002", status="paid", amount=12.50),
    Row(order_id="O90003", customer_id="C0003", status="cancelled", amount=80.00),
    Row(order_id="O90004", customer_id="C0001", status="paid", amount=23.10),
    Row(order_id="O90005", customer_id="C0004", status="pending", amount=5.00),
]
df = spark.createDataFrame(pedidos)
df.printSchema()
df.show()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)

+--------+-----------+---------+------+
|order_id|customer_id|   status|amount|
+--------+-----------+---------+------+
|  O90001|      C0001|     paid|  49.9|
|  O90002|      C0002|     paid|  12.5|
|  O90003|      C0003|cancelled|  80.0|
|  O90004|      C0001|     paid|  23.1|
|  O90005|      C0004|  pending|   5.0|
+--------+-----------+---------+------+



Ejecutamos filter para ver que no hace nada hasta que no pones el count

In [14]:
paid = df.filter(df.status == "paid")
print("después del filter, Spark aún no ha contado nada")
print("paid count =", paid.count())


después del filter, Spark aún no ha contado nada
paid count = 3


Vemos el query plan

In [19]:
paid.explain("formatted")
paid.show()

== Physical Plan ==
* Filter (2)
+- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [4]: [order_id#34, customer_id#35, status#36, amount#37]
Arguments: [order_id#34, customer_id#35, status#36, amount#37], MapPartitionsRDD[16] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [4]: [order_id#34, customer_id#35, status#36, amount#37]
Condition : (isnotnull(status#36) AND (status#36 = paid))


+--------+-----------+------+------+
|order_id|customer_id|status|amount|
+--------+-----------+------+------+
|  O90001|      C0001|  paid|  49.9|
|  O90002|      C0002|  paid|  12.5|
|  O90004|      C0001|  paid|  23.1|
+--------+-----------+------+------+



Ejemplo añador columna

In [21]:
from pyspark.sql.functions import lit

df.withColumn("channel", lit("web")).select("order_id", "channel").show()

+--------+-------+
|order_id|channel|
+--------+-------+
|  O90001|    web|
|  O90002|    web|
|  O90003|    web|
|  O90004|    web|
|  O90005|    web|
+--------+-------+



In [22]:
df.show()

+--------+-----------+---------+------+
|order_id|customer_id|   status|amount|
+--------+-----------+---------+------+
|  O90001|      C0001|     paid|  49.9|
|  O90002|      C0002|     paid|  12.5|
|  O90003|      C0003|cancelled|  80.0|
|  O90004|      C0001|     paid|  23.1|
|  O90005|      C0004|  pending|   5.0|
+--------+-----------+---------+------+

